## Notes

Context information
- can travel for 0.25 sq km in 1hr. We assume this is the flight time minimum too.
- each full sq km costs Rs18,000. This makes minimum assumed flight cost to be Rs4,500
- The drone company groups any buildings within 500m of each other, creates the convex hull, and adds a 20m buffer.

## Setup

In [ ]:
INDIA_PROJECTED_CRS = "24378"

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

In [ ]:
from gridsample.utils import save_shapefiles
# from gridsample.mapping.plot import create_interactive_map

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
OUTPUT_DATA_ROOT_DIR = DATA_DIR / "01_processed" / "Goverment Buildings"
OUTPUT_DATA_ROOT_DIR.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
DISTRICT_NAME = "Seoni"
DISTRICT_FILENAME = "Seoni_20 kVA and above contract demand_final.csv"
# Note: this CSV was manually exported and formatted from the original Excel file of the same name
df = pd.read_csv(RAW_DATA_DIR / "government_buildings" / DISTRICT_FILENAME)
OUTPUT_DATA_DIR = OUTPUT_DATA_ROOT_DIR / DISTRICT_NAME
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Note: "Consolidated data of government buildings 24 districts.xlsx" is the file with all 24 districts on it

In [ ]:
df.columns

In [ ]:
df.rename(
    columns={
        "Latitude ": "lat",
        "Longitude": "lon",
    },
    inplace=True,
)

# drop any rows that have nulls in lat or lon
df = df.dropna(subset=["lat", "lon"])

In [ ]:
# convert to gdf
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326"
)

In [ ]:
gdf.head()

In [ ]:
gdf.plot(markersize=1)

In [ ]:
# save the gdf to a shapefile
save_shapefiles(
    gdf,
    OUTPUT_DATA_DIR,
    f"{DISTRICT_NAME} processed buildings",
    formats=["parquet", "kml"],
)

## Clustering

### Cluster buildings at different radii

In [ ]:
from utils import (
    build_graph_from_gdf_with_distance_threshold,
    get_connected_components_by_distance_threshold,
)

In [ ]:
gdf_with_cluster_labels = gdf.copy().to_crs(INDIA_PROJECTED_CRS)
cluster_col_names = []

G = build_graph_from_gdf_with_distance_threshold(
    gdf_with_cluster_labels, distance_threshold=5_0000
)
thresholds = [500, 1_000, 2_000, 3_000]
threshold_names = ["500m", "1km", "2km", "3km"]

for threshold, threshold_str in zip(thresholds, threshold_names):
    cluster_col_name = f"cluster_id_{threshold_str}"
    cluster_col_names.append(cluster_col_name)

    cluster_labels_df, G_filtered_with_cluster_labels = (
        get_connected_components_by_distance_threshold(
            G,
            distance_threshold=threshold,
            cluster_id_col_name=cluster_col_name,
            cluster_id_prefix="SHAPE_",
        )
    )

    # add column to gdf based on index match
    gdf_with_cluster_labels[cluster_col_name] = gdf_with_cluster_labels.index.map(
        cluster_labels_df[cluster_col_name]
    )

In [ ]:
gdf_with_cluster_labels.head()

In [ ]:
# save to file
save_shapefiles(
    gdf_with_cluster_labels,
    OUTPUT_DATA_DIR,
    f"{DISTRICT_NAME} processed buildings with clusters",
    formats=["csv", "parquet", "kml"],
)

### For every cluster radius, save cluster boundaries with stats

In [ ]:
CLUSTER_BOUNDARY_BUFFER = 20 # in meters

In [ ]:
def create_cluster_summary(gdf_with_cluster_labels, cluster_col_name="cluster"):
    """
    Create a summary GeoDataFrame of clusters with area and cost calculations.

    Args:
        gdf_with_cluster_labels: GeoDataFrame with cluster labels
        cluster_col_name: Name of the column containing cluster labels

    Returns:
        GeoDataFrame with cluster metrics
    """
    # Calculate building counts and create convex hulls
    building_counts = gdf_with_cluster_labels.groupby(cluster_col_name).size()
    hulls = gdf_with_cluster_labels.dissolve(cluster_col_name).convex_hull.buffer(
        CLUSTER_BOUNDARY_BUFFER
    )

    # Create cluster summary DataFrame
    cluster_summary = gpd.GeoDataFrame(
        {"Building Count": building_counts, "geometry": hulls}, crs=INDIA_PROJECTED_CRS
    ).reset_index()

    # Rename cluster ID column
    cluster_summary = cluster_summary.rename(columns={cluster_col_name: "shape_id"})

    # Calculate derived metrics
    cluster_summary["Original Area (km²)"] = round(cluster_summary.area / 1_000_000, 4)
    # enforce assumed minimum flight area and cost
    cluster_summary["Adjusted Area (km²)"] = cluster_summary[
        "Original Area (km²)"
    ].apply(lambda x: 0.25 if x < 0.25 else x)
    cluster_summary["Charged Flight Time (hours)"] = round(
        cluster_summary["Adjusted Area (km²)"] / 0.25, 4
    )
    cluster_summary["Cost (INR)"] = round(
        cluster_summary["Adjusted Area (km²)"] * 18_000, 0
    )

    return cluster_summary

In [ ]:
cluster_gdf_list = []

for threshold_str in threshold_names:
    cluster_col_name = f"cluster_id_{threshold_str}"

    cluster_gdf = create_cluster_summary(
        gdf_with_cluster_labels, cluster_col_name=cluster_col_name
    )
    cluster_gdf_list.append(cluster_gdf)

    # save cluster-level data to file
    save_shapefiles(
        cluster_gdf,
        OUTPUT_DATA_DIR / threshold_str,
        f"{threshold_str}_cluster_boundaries_with_stats",
        formats=["csv", "kml"],
    )

    # Plot the clusters
    total_area = cluster_gdf["Original Area (km²)"].sum()
    totla_charged_area = cluster_gdf["Adjusted Area (km²)"].sum()
    total_cost = cluster_gdf["Cost (INR)"].sum()

    ax = cluster_gdf.plot(
        figsize=(10, 10),
        color="black",
        alpha=0.5,
    )

    gdf_with_cluster_labels.plot(
        ax=ax,
        column=cluster_col_name,
        cmap="tab20",
        markersize=5,
        legend=True,
    )

    # add a 10km line to show scale on the plot
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    length = 10000
    length_name = "10km"
    ax.plot(
        [xmax - length, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-"
    )
    ax.plot(
        [xmax - length, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-"
    )
    ax.text(xmax - length / 2, ymin + 400, length_name, fontsize=6, ha="center")

    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_title(f"{threshold_str} Grouping Threshold")

    # add text to show total area and cost
    ax.text(
        xmax - 45000,
        ymax - 8000,
        (
            f"Total Area: {total_area:.2f}km²\n"
            f"Total Charged Area: {totla_charged_area:.2f}km²\n"
            f"Total Cost: {total_cost:,.0f} INR"
        ),
        fontsize=12,
        ha="left",
    )
    # remove extra white space in saved file
    plt.tight_layout()
    plt.savefig(OUTPUT_DATA_DIR / threshold_str / f"clustered_buildings_{threshold_str}.png", dpi=300)
    plt.show()

    # # Save original data with added column for selected cluster IDs
    # # Get all cluster columns except the current one
    # cluster_cols_to_drop = [
    #     col_name for col_name in cluster_col_names if col_name != cluster_col_name
    # ]
    # save_shapefiles(
    #     gdf_with_cluster_labels.drop(columns=cluster_cols_to_drop),
    #     OUTPUT_DATA_DIR / threshold_str,
    #     f"buildings_with_{threshold_str}_cluster_ids",
    #     formats=["csv", "kml"],
    # )

### Make summary table with one row per clustering radius and stats

In [ ]:
# concatenate the cluster_gdf_list with the threshold names added
all_groupings_gdf = pd.concat(
    [cluster_gdf.assign(threshold=threshold) for cluster_gdf, threshold in zip(cluster_gdf_list, threshold_names)]
)

In [ ]:
# make a summary of the total area total charged area and cost and total number of shapes for each threshold
threshold_summary = all_groupings_gdf.groupby("threshold").agg(
    {
        "Original Area (km²)": "sum",
        "Adjusted Area (km²)": "sum",
        "Cost (INR)": "sum",
        "shape_id": "count",
    }
)
# drop rename shape_id to Number of Shapes
threshold_summary = threshold_summary.rename(columns={"shape_id": "Flights"})

# add number of single-building flights
threshold_summary["Single-building Flights"] = all_groupings_gdf[
    all_groupings_gdf["Building Count"] == 1
].groupby("threshold")["shape_id"].count()

In [ ]:
# sort values by threshold ID but make sure we order correctly by the numbers not 10, 1, 20, 2 etc
threshold_summary = threshold_summary.reset_index()
threshold_summary["threshold"] = pd.Categorical(threshold_summary["threshold"], threshold_names)
threshold_summary = threshold_summary.sort_values("threshold")

In [ ]:
# round areas to 2 decimal places and cost as int
threshold_summary["Original Area (km²)"] = threshold_summary["Original Area (km²)"].round(2)
threshold_summary["Adjusted Area (km²)"] = threshold_summary["Adjusted Area (km²)"].round(2)
threshold_summary["Cost (INR)"] = threshold_summary["Cost (INR)"].astype(int)

In [ ]:
threshold_summary

In [ ]:
# Display the pivot table
threshold_summary.to_csv(OUTPUT_DATA_DIR / "all_clustering_stats.csv", index=False)

## Scraps

### Other clustering methods

In [ ]:
# # cluster with kmeans
# from sklearn.cluster import KMeans

# kmeans = KMeans(n_clusters=8, random_state=0)
# gdf["cluster"] = [str(i) for i in kmeans.fit_predict(gdf[["lon", "lat"]])]
# gdf.plot(column="cluster", markersize=1, legend=True)

In [ ]:
# from sklearn.cluster import HDBSCAN

# dbscan = HDBSCAN(n_jobs=-1)
# gdf["cluster"] = dbscan.fit_predict(gdf[["lon", "lat"]])
# gdf.plot(column="cluster", markersize=1, legend=True)

### Travelling Salesman

In [ ]:
# gdf_projected = gdf.to_crs("EPSG:24378")

In [ ]:
# gdf_projected.lat = gdf_projected.geometry.y
# gdf_projected.lon = gdf_projected.geometry.x

In [ ]:
# from scipy.spatial import distance_matrix

# # Calculate the distance matrix
# dist_matrix = distance_matrix(gdf_projected[["lon", "lat"]], gdf_projected[["lon", "lat"]])

In [ ]:
# from ortools.constraint_solver import pywrapcp
# from ortools.constraint_solver import routing_enums_pb2

In [ ]:
# # make list of latlon tuples
# lonlat = [(lon, lat) for lon, lat in zip(gdf_projected.lon, gdf_projected.lat)]

In [ ]:
# from scipy.spatial import distance_matrix
# from ortools.constraint_solver import pywrapcp
# from ortools.constraint_solver import routing_enums_pb2

# # Coordinates example
# coordinates = lonlat

# # Distance matrix
# dist_matrix = distance_matrix(coordinates, coordinates)

# def create_data_model():
#     return {'distance_matrix': dist_matrix, 'num_locations': len(coordinates)}

# data = create_data_model()
# manager = pywrapcp.RoutingIndexManager(data['num_locations'], 1, 0)
# routing = pywrapcp.RoutingModel(manager)

# def distance_callback(from_index, to_index):
#     from_node = manager.IndexToNode(from_index)
#     to_node = manager.IndexToNode(to_index)
#     return int(data['distance_matrix'][from_node][to_node])

# transit_callback_index = routing.RegisterTransitCallback(distance_callback)
# routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# search_parameters = pywrapcp.DefaultRoutingSearchParameters()
# search_parameters.first_solution_strategy = (
#     routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
# search_parameters.local_search_metaheuristic = (
#     routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
# search_parameters.time_limit.seconds = 30

# solution = routing.SolveWithParameters(search_parameters)

In [ ]:
# # Print the solution (ordered route)
# if solution:
#     index = routing.Start(0)
#     route = []
#     while not routing.IsEnd(index):
#         route.append(manager.IndexToNode(index))
#         index = solution.Value(routing.NextVar(index))
#     route.append(manager.IndexToNode(index))
#     print("Optimal Path:", route)

In [ ]:
# # make a multi-line string
# from shapely.geometry import LineString

In [ ]:
# line_gdf = gpd.GeoDataFrame(
#     data={"start_index": route[:-1], "end_index": route[1:]},
#     geometry=[
#         LineString(
#             [
#                 gdf_projected.iloc[route[i]].geometry,
#                 gdf_projected.iloc[route[i + 1]].geometry,
#             ]
#         )
#         for i in range(len(route) - 1)
#     ],
# )

In [ ]:
# line_gdf

In [ ]:
# # add start and end cluster names
# line_gdf["start_cluster"] = gdf_projected.iloc[line_gdf.start_index].cluster.values
# line_gdf["end_cluster"] = gdf_projected.iloc[line_gdf.end_index].cluster.values

In [ ]:
# filtered_line_gdf = line_gdf[line_gdf["start_cluster"] == line_gdf["end_cluster"]]

In [ ]:
# filtered_line_gdf.length.sum()

In [ ]:
# fig, ax = plt.subplots(figsize=(15,15))
# gdf_projected.plot(ax=ax, column="cluster", markersize=35)
# # line_gdf.plot(ax=ax)
# filtered_line_gdf.plot(ax=ax, color="black")

# # add a 1km line to show scale on the plot
# xmin, xmax = ax.get_xlim()
# ymin, ymax = ax.get_ylim()
# length = 10000
# length_name = "10km"
# ax.plot([xmax - length, xmax], [ymin, ymin], color="white", linewidth=5, linestyle="-")
# ax.plot([xmax - length, xmax], [ymin, ymin], color="black", linewidth=2, linestyle="-")
# ax.text(xmax - length/2, ymin + 400, length_name, fontsize=6, ha="center")

# # axes off
# ax.axis("off")

In [ ]:
# flight_paths_gdf = filtered_line_gdf.dissolve("start_cluster").reset_index()[["start_cluster", "geometry"]]
# flight_paths_gdf.rename(columns={"start_cluster": "cluster"}, inplace=True)
# flight_paths_gdf.set_crs("EPSG:24378", inplace=True)

In [ ]:
# # get count of buildings in each cluster
# n_buildings_per_cluster = gdf_projected.groupby("cluster").size()
# flight_paths_gdf["Number of Buildings"] = flight_paths_gdf["cluster"].map(n_buildings_per_cluster)

In [ ]:
# flight_paths_gdf["Length (km)"] = round(flight_paths_gdf.length / 1000, 2)

In [ ]:
# line_gdf[line_gdf["start_cluster"] != line_gdf["end_cluster"]].length

In [ ]:
# save_shapefiles(
#     flight_paths_gdf.to_crs("EPSG:4326"),
#     OUTPUT_DATA_DIR,
#     "flight_paths_grouped",
#     formats=["parquet", "kml"],
# )

In [ ]:
# save_shapefiles(
#     gdf.to_crs("EPSG:4326"),
#     OUTPUT_DATA_DIR,
#     "gov_buildings_grouped",
#     formats=["parquet", "kml"],
# )

In [ ]:
# flight_path_buffered_gdf = flight_paths_gdf.copy()
# flight_path_buffered_gdf["geometry"] = flight_path_buffered_gdf.buffer(50)

# save_shapefiles(
#     flight_path_buffered_gdf.to_crs("EPSG:4326"),
#     OUTPUT_DATA_DIR,
#     "flight_paths_grouped_buffered",
#     formats=["parquet", "kml"],
# )